# Etape 1 - Mise en place de l'environnement
## Puls-Events RAG System

Objectif : Installer toutes les dependances et verifier que les modules cles fonctionnent.

| Composant | Role |
|-----------|------|
| faiss-cpu | Base vectorielle |
| langchain | Orchestration RAG |
| sentence-transformers | Embeddings |
| mistralai | LLM |
| fastapi | API REST |

## 1. Installation des dependances

In [2]:
# Desinstallation de tout ce qui est en conflit
#!pip uninstall -y langchain langchain-community langchain-core langchain-mistralai pydantic

# Reinstallation avec des versions compatibles testees
!pip install -q \
    "langchain==0.1.20" \
    "langchain-community==0.0.38" \
    "langchain-core==0.1.52" \
    "langchain-mistralai==0.1.4" \
    "pydantic==1.10.21" \
    "faiss-cpu" \
    "sentence-transformers"

In [3]:
!pip install -q --force-reinstall numpy==2.0.2
import importlib, sys
# Vider le cache des modules numpy deja charges
for mod in list(sys.modules.keys()):
    if 'numpy' in mod:
        del sys.modules[mod]

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.38 requires numpy<2,>=1, but you have numpy 2.0.2 which is incompatible.
langchain 0.1.20 requires numpy<2,>=1, but you have numpy 2.0.2 which is incompatible.
albumentations 2.0.8 requires pydantic>=2.9.2, but you have pydantic 1.10.21 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
thinc 8.3.13 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.21 which is incompatible.
spacy 3.8.14 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.21 which is incompatible.
db-dtypes 1.5.1 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 1.10.21 which is incompatible.
bigquery-magics 0.14.0 requires packaging>=24.2.0, but

## 2. Cle API Mistral

Ajouter la cle dans le menu Secrets de Colab (icone cle a gauche) sous le nom `MISTRAL_API_KEY`

In [4]:
import os

try:
    from google.colab import userdata
    os.environ['MISTRAL_API_KEY'] = userdata.get('MISTRAL_API_KEY')
    print('Cle Mistral chargee depuis Colab Secrets')
except Exception:
    from getpass import getpass
    os.environ['MISTRAL_API_KEY'] = getpass('Entrez votre cle API Mistral : ')
    print('Cle Mistral definie manuellement')

Cle Mistral chargee depuis Colab Secrets


## 3. Test des imports

In [5]:
# Test FAISS
try:
    import faiss
    index = faiss.IndexFlatL2(128)
    assert index.ntotal == 0
    print('OK faiss version:', faiss.__version__)
except ImportError as e:
    print('ERREUR faiss:', e)

OK faiss version: 1.13.2


/usr/local/lib/python3.12/dist-packages/faiss/__init__.py:11: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  import numpy as np


In [6]:
# Test Embeddings
try:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    print('Chargement du modele embeddings (1-2 min)...')
    embeddings = HuggingFaceEmbeddings(
        model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        model_kwargs={'device': 'cpu'}
    )
    test_vec = embeddings.embed_query('Concert de jazz a Paris')
    print('OK embeddings, dimension:', len(test_vec))
except Exception as e:
    print('ERREUR embeddings:', e)

Chargement du modele embeddings (1-2 min)...


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


ERREUR embeddings: Failed to import transformers.trainer_callback because of the following error (look up to see its traceback):
module 'numpy.dtypes' has no attribute 'UInt32DType'


In [5]:
!pip install langchain-mistralai

In [7]:
from langchain_mistralai import ChatMistralAI
import os

llm = ChatMistralAI(
    model="mistral-small-latest",
    mistral_api_key=os.environ["MISTRAL_API_KEY"]
)

response = llm.invoke("Dis bonjour en une phrase.")
print(response.content)

Bonjour ! 😊


In [8]:
# Test FastAPI
try:
    import fastapi, uvicorn
    print('OK fastapi', fastapi.__version__)
    print('OK uvicorn', uvicorn.__version__)
except ImportError as e:
    print('ERREUR fastapi:', e)

ERREUR fastapi: cannot import name 'IncEx' from 'pydantic.main' (/usr/local/lib/python3.12/dist-packages/pydantic/main.cpython-312-x86_64-linux-gnu.so)


## 4. Mini pipeline RAG - test d'integration

In [10]:
import requests, os
import numpy as np

MISTRAL_API_KEY = os.environ['MISTRAL_API_KEY']

def get_embeddings(texts):
    """Embeddings via API Mistral - pas besoin de sentence-transformers"""
    if isinstance(texts, str):
        texts = [texts]
    r = requests.post(
        "https://api.mistral.ai/v1/embeddings",
        headers={"Authorization": f"Bearer {MISTRAL_API_KEY}", "Content-Type": "application/json"},
        json={"model": "mistral-embed", "input": texts}
    )
    return [item['embedding'] for item in r.json()['data']]

class MistralEmbeddings:
    def embed_documents(self, texts):
        return get_embeddings(texts)
    def embed_query(self, text):
        return get_embeddings([text])[0]

embeddings = MistralEmbeddings()

# Test
test = embeddings.embed_query("concert jazz Paris")
print(f'OK embeddings Mistral, dimension : {len(test)}')

OK embeddings Mistral, dimension : 1024


In [11]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from typing import List

class MistralEmbeddings(Embeddings):
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return get_embeddings(texts)
    def embed_query(self, text: str) -> List[float]:
        return get_embeddings([text])[0]

embeddings = MistralEmbeddings()

# Recree le vectorstore avec la nouvelle classe
sample_events = [
    Document(page_content='Concert de jazz au Cafe de la Danse, Paris 11e. Le 15 mars 2025, 20h.', metadata={'type': 'concert', 'ville': 'Paris'}),
    Document(page_content='Exposition Monet au Musee Marmottan, Paris 16e. Du 10 fevrier au 30 juin 2025.', metadata={'type': 'exposition', 'ville': 'Paris'}),
    Document(page_content='Festival de danse contemporaine a Lyon. Du 5 au 20 avril 2025.', metadata={'type': 'festival', 'ville': 'Lyon'}),
]

vectorstore = FAISS.from_documents(sample_events, embeddings)
print('Index FAISS cree avec', vectorstore.index.ntotal, 'vecteurs')

query = "Quels concerts ont lieu a Paris ?"
docs = vectorstore.similarity_search(query, k=2)
print('Documents recuperes :')
for i, doc in enumerate(docs):
    print(f'  {i+1}:', doc.page_content)

Index FAISS cree avec 3 vecteurs
Documents recuperes :
  1: Concert de jazz au Cafe de la Danse, Paris 11e. Le 15 mars 2025, 20h.
  2: Festival de danse contemporaine a Lyon. Du 5 au 20 avril 2025.


## 5. Resume de l'environnement

In [12]:
import sys, platform
import langchain, sentence_transformers

print('=' * 45)
print('RESUME ENVIRONNEMENT')
print('=' * 45)
print('Python         :', sys.version.split()[0])
print('Plateforme     :', platform.platform())
print('faiss-cpu      :', faiss.__version__)
print('langchain      :', langchain.__version__)
print('sentence-transf:', sentence_transformers.__version__)
print('fastapi        :', fastapi.__version__)
print('=' * 45)
print('Etape 1 terminee - Environnement pret !')

RuntimeError: Failed to import transformers.trainer_callback because of the following error (look up to see its traceback):
cannot import name 'dispatch_model' from partially initialized module 'accelerate.big_modeling' (most likely due to a circular import) (/usr/local/lib/python3.12/dist-packages/accelerate/big_modeling.py)

# Etape 2 - Collecte des donnees Open Agenda


In [13]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import re
import os
from google.colab import userdata

os.environ['OPENAGENDA_API_KEY'] = userdata.get('OPENAGENDA_API_KEY')
API_KEY = os.environ.get('OPENAGENDA_API_KEY', '')

# Configuration
BASE_URL = 'https://api.openagenda.com/v2'

# Parametres de filtrage
VILLE_CIBLE = 'Paris'
DATE_DEBUT = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')
DATE_FIN = (datetime.now() + timedelta(days=365)).strftime('%Y-%m-%d')

print(f'Zone ciblee   : {VILLE_CIBLE}')
print(f'Periode debut : {DATE_DEBUT}')
print(f'Periode fin   : {DATE_FIN}')
print(f'Cle API       : {"definie" if API_KEY else "non definie (mode public)"}')

Zone ciblee   : Paris
Periode debut : 2025-05-15
Periode fin   : 2027-05-15
Cle API       : definie


In [14]:
# On abandonne OpenAgenda et on utilise l'API officielle de la Ville de Paris
# Dataset "Que faire à Paris" - totalement gratuit, pas de clé requise

import requests
import pandas as pd
from datetime import datetime, timedelta
import re

BASE_URL_PARIS = 'https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/que-faire-a-paris-/records'

DATE_DEBUT = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%dT%H:%M:%S')
DATE_FIN = (datetime.now() + timedelta(days=365)).strftime('%Y-%m-%dT%H:%M:%S')

def fetch_events_paris(limit=100, max_records=500):
    all_records = []
    offset = 0

    while True:
        params = {
            'limit': limit,
            'offset': offset,
            'where': f'date_start >= "{DATE_DEBUT}" AND date_start <= "{DATE_FIN}"',
            'order_by': 'date_start ASC',
            'lang': 'fr',
        }
        r = requests.get(BASE_URL_PARIS, params=params, timeout=30)

        if r.status_code != 200:
            print(f'Erreur {r.status_code}:', r.text[:100])
            break

        data = r.json()
        records = data.get('results', [])

        if not records:
            break

        all_records.extend(records)
        print(f'Offset {offset} : {len(records)} evenements (total: {len(all_records)})')

        offset += limit
        if len(all_records) >= max_records:
            break

    print(f'\nTotal : {len(all_records)} evenements recuperes')
    return all_records

raw_events = fetch_events_paris()

Offset 0 : 100 evenements (total: 100)
Offset 100 : 100 evenements (total: 200)
Offset 200 : 100 evenements (total: 300)
Offset 300 : 100 evenements (total: 400)
Offset 400 : 100 evenements (total: 500)

Total : 500 evenements recuperes


In [15]:
def clean_html(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_event_fields(event):
    return {
        'uid': str(event.get('id', '')),
        'titre': clean_html(event.get('title', '') or ''),
        'description': clean_html(event.get('description', '') or event.get('lead_text', '') or ''),
        'lieu': event.get('address_name', '') or '',
        'adresse': event.get('address_street', '') or '',
        'ville': 'Paris',
        'code_postal': event.get('address_zipcode', '') or '',
        'date_debut': (event.get('date_start', '') or '')[:10],
        'date_fin': (event.get('date_end', '') or '')[:10],
        'categories': event.get('category', '') or '',
        'url': event.get('url', '') or '',
    }

# Extraction
df = pd.DataFrame([extract_event_fields(e) for e in raw_events])

# Nettoyage
df = df.drop_duplicates(subset='uid')
df = df[df['description'].str.len() > 20]
df = df[df['titre'].str.len() > 2]

# Champ texte pour la vectorisation
df['texte_complet'] = (
    'Evenement : ' + df['titre'] + '. ' +
    'Lieu : ' + df['lieu'] + ', ' + df['ville'] + '. ' +
    'Du ' + df['date_debut'] + ' au ' + df['date_fin'] + '. ' +
    'Categories : ' + df['categories'] + '. ' +
    'Description : ' + df['description'].str[:500]
)

print(f'Dataset final : {df.shape}')
print('\nExemple texte_complet :')
print(df['texte_complet'].iloc[0])

Dataset final : (498, 12)

Exemple texte_complet :
Evenement : Requiem de Verdi Eglise de la Madeleine. Lieu : Eglise de la Madeleine, Paris. Du 2025-05-30 au 2026-11-07. Categories : . Description : Requiem de Verdi Orchestre : Orchestre Hélios Direction : Romain Dumas Chœur Éphémère de Paris Présentation de l'oeuvre Le Requiem de Verdi, un « opéra de la mort » ? Le 22 mai 1873, le poète Alessandro Manzoni meurt. Verdi est bouleversé par la disparition du poète, à qui il avait adressé une dédicace en 1867 « je vous estime et vous vénère autant qu’on peut estimer et vénérer sur cette terre, et comme homme et comme incarnant le véritable honneur de notre patrie tourmentée ». Ainsi le composit


In [16]:
print('=== ANALYSE DU DATASET ===')
print(f'Nombre evenements : {len(df)}')
print(f'Periode           : {df["date_debut"].min()} -> {df["date_debut"].max()}')
print(f'Longueur desc moy : {df["description"].str.len().mean():.0f} caracteres')
print(f'\nApercu :')
df[['titre', 'ville', 'date_debut', 'categories']].head(10)

=== ANALYSE DU DATASET ===
Nombre evenements : 498
Periode           : 2025-05-30 -> 2026-03-14
Longueur desc moy : 1054 caracteres

Apercu :


,titre,ville,date_debut,categories
0,Requiem de Verdi Eglise de la Madeleine,Paris,2025-05-30,
1,Requiem de Mozart & Boléro de Ravel,Paris,2025-05-31,
2,« Mon premier atelier santé environnement » : ...,Paris,2025-06-16,
3,Transparence : la première exposition du Palai...,Paris,2025-06-20,
4,Le café info séniors des mardis,Paris,2025-07-08,
5,Atelier de conversation Français Langue Etrangère,Paris,2025-07-11,
6,Toutes les expositions gratuites de la Ville,Paris,2025-08-01,
7,"Foot, basket, rugby, hand : où encourager les ...",Paris,2025-08-17,
8,La bibliothèque Marguerite Audoux cultive son ...,Paris,2025-08-21,
9,La bibliothèque Marguerite Audoux fait son cinéma,Paris,2025-08-21,


In [17]:
os.makedirs('/content/data', exist_ok=True)
df.to_csv('/content/data/events_clean.csv', index=False, encoding='utf-8')
df.to_json('/content/data/events_clean.json', orient='records', force_ascii=False, indent=2)
print(f'Sauvegarde OK : {len(df)} evenements dans /content/data/')

Sauvegarde OK : 498 evenements dans /content/data/


In [18]:
print('=== TESTS UNITAIRES ETAPE 2 ===')

# Test 1
try:
    assert len(df) > 0
    print('OK  Dataset non vide :', len(df), 'evenements')
except AssertionError:
    print('FAIL Dataset vide')

# Test 2
try:
    for col in ['uid', 'titre', 'description', 'ville', 'date_debut', 'texte_complet']:
        assert col in df.columns
    print('OK  Colonnes obligatoires presentes')
except AssertionError as e:
    print('FAIL Colonne manquante :', e)

# Test 3
try:
    assert df['uid'].nunique() == len(df)
    print('OK  Pas de doublons')
except AssertionError:
    print('FAIL Doublons detectes')

# Test 4
try:
    pct_vide = (df['description'].str.len() <= 20).mean()
    assert pct_vide < 0.1
    print(f'OK  Descriptions valides ({pct_vide:.0%} vides)')
except AssertionError:
    print(f'FAIL Trop de descriptions vides : {pct_vide:.0%}')

# Test 5
try:
    assert any(VILLE_CIBLE.lower() in v.lower() for v in df['ville'].unique())
    print(f'OK  Ville cible "{VILLE_CIBLE}" presente')
except AssertionError:
    print(f'FAIL Ville "{VILLE_CIBLE}" absente')

# Test 6
try:
    dates = pd.to_datetime(df['date_debut'], errors='coerce').dropna()
    assert len(dates) > 0
    assert dates.min() >= pd.Timestamp(datetime.now() - timedelta(days=370))
    print(f'OK  Dates valides ({dates.min().date()} -> {dates.max().date()})')
except AssertionError as e:
    print('FAIL Dates invalides :', e)

# Test 7
try:
    moy = df['texte_complet'].str.len().mean()
    assert moy > 100
    print(f'OK  Texte complet assez long (moy {moy:.0f} chars)')
except AssertionError:
    print(f'FAIL Texte trop court : {moy:.0f} chars')

print('\n=== FIN DES TESTS ===')

=== TESTS UNITAIRES ETAPE 2 ===
OK  Dataset non vide : 498 evenements
OK  Colonnes obligatoires presentes
OK  Pas de doublons
OK  Descriptions valides (0% vides)
OK  Ville cible "Paris" presente
OK  Dates valides (2025-05-30 -> 2026-03-14)
OK  Texte complet assez long (moy 608 chars)

=== FIN DES TESTS ===


# Etape 3 - Vectorisation et indexation FAISS

In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Decoupage des textes en chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

# Creation des chunks avec metadonnees
chunks = []
for _, row in df.iterrows():
    splits = text_splitter.split_text(row['texte_complet'])
    for chunk in splits:
        chunks.append({
            'texte': chunk,
            'uid': row['uid'],
            'titre': row['titre'],
            'lieu': row['lieu'],
            'ville': row['ville'],
            'date_debut': row['date_debut'],
            'date_fin': row['date_fin'],
            'categories': row['categories'],
            'url': row['url'],
        })

print(f'Nombre de chunks : {len(chunks)}')
print(f'Moyenne chunks par evenement : {len(chunks)/len(df):.1f}')
print(f'\nExemple chunk :')
print(chunks[0]['texte'])

Nombre de chunks : 957
Moyenne chunks par evenement : 1.9

Exemple chunk :
Evenement : Requiem de Verdi Eglise de la Madeleine. Lieu : Eglise de la Madeleine, Paris. Du 2025-05-30 au 2026-11-07. Categories : . Description : Requiem de Verdi Orchestre : Orchestre Hélios Direction : Romain Dumas Chœur Éphémère de Paris Présentation de l'oeuvre Le Requiem de Verdi, un « opéra de la mort » ? Le 22 mai 1873, le poète Alessandro Manzoni meurt


In [22]:
import requests, os
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from typing import List

# Embeddings via API Mistral - pas besoin de sentence-transformers
class MistralEmbeddings(Embeddings):
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        r = requests.post(
            "https://api.mistral.ai/v1/embeddings",
            headers={"Authorization": f"Bearer {os.environ['MISTRAL_API_KEY']}", "Content-Type": "application/json"},
            json={"model": "mistral-embed", "input": texts}
        )
        return [item['embedding'] for item in r.json()['data']]

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]

embeddings_model = MistralEmbeddings()
print('OK embeddings Mistral pret !')

# Conversion en Documents LangChain
documents = [
    Document(
        page_content=chunk['texte'],
        metadata={
            'uid': chunk['uid'],
            'titre': chunk['titre'],
            'lieu': chunk['lieu'],
            'ville': chunk['ville'],
            'date_debut': chunk['date_debut'],
            'date_fin': chunk['date_fin'],
            'categories': chunk['categories'],
            'url': chunk['url'],
        }
    )
    for chunk in chunks
]

print(f'Documents prets : {len(documents)}')

OK embeddings Mistral pret !
Documents prets : 957


In [23]:
from langchain_community.vectorstores import FAISS
import time

print('Construction de l index FAISS...')
start = time.time()

# Indexation par batch pour eviter les timeouts
BATCH_SIZE = 50
vectorstore = None

for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i:i+BATCH_SIZE]
    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embeddings_model)
    else:
        vectorstore.add_documents(batch)
    print(f'  Batch {i//BATCH_SIZE + 1}/{(len(documents)-1)//BATCH_SIZE + 1} indexe ({i+len(batch)}/{len(documents)} chunks)')

elapsed = time.time() - start
print(f'\nIndex FAISS construit en {elapsed:.1f}s')
print(f'Nombre de vecteurs dans l index : {vectorstore.index.ntotal}')

Construction de l index FAISS...
  Batch 1/20 indexe (50/957 chunks)
  Batch 2/20 indexe (100/957 chunks)
  Batch 3/20 indexe (150/957 chunks)
  Batch 4/20 indexe (200/957 chunks)
  Batch 5/20 indexe (250/957 chunks)
  Batch 6/20 indexe (300/957 chunks)
  Batch 7/20 indexe (350/957 chunks)
  Batch 8/20 indexe (400/957 chunks)
  Batch 9/20 indexe (450/957 chunks)
  Batch 10/20 indexe (500/957 chunks)
  Batch 11/20 indexe (550/957 chunks)
  Batch 12/20 indexe (600/957 chunks)
  Batch 13/20 indexe (650/957 chunks)
  Batch 14/20 indexe (700/957 chunks)
  Batch 15/20 indexe (750/957 chunks)
  Batch 16/20 indexe (800/957 chunks)
  Batch 17/20 indexe (850/957 chunks)
  Batch 18/20 indexe (900/957 chunks)
  Batch 19/20 indexe (950/957 chunks)
  Batch 20/20 indexe (957/957 chunks)

Index FAISS construit en 19.9s
Nombre de vecteurs dans l index : 957


In [24]:
import os

# Sauvegarde de l'index FAISS
os.makedirs('/content/vectorstore', exist_ok=True)
vectorstore.save_local('/content/vectorstore/faiss_index')

print('Index sauvegarde dans /content/vectorstore/faiss_index/')
print('Fichiers crees :')
for f in os.listdir('/content/vectorstore/faiss_index'):
    taille = os.path.getsize(f'/content/vectorstore/faiss_index/{f}')
    print(f'  {f} ({taille/1024:.1f} Ko)')

Index sauvegarde dans /content/vectorstore/faiss_index/
Fichiers crees :
  index.faiss (3828.0 Ko)
  index.pkl (527.9 Ko)


In [25]:
# Tests de recherche semantique
print('=== TESTS DE RECHERCHE SEMANTIQUE ===\n')

requetes_test = [
    'concert de musique classique',
    'exposition peinture moderne',
    'spectacle enfants famille',
    'theatre comedie',
    'evenement gratuit weekend',
]

for requete in requetes_test:
    print(f'Requete : "{requete}"')
    results = vectorstore.similarity_search(requete, k=2)
    for i, doc in enumerate(results):
        print(f'  {i+1}. {doc.metadata["titre"]} — {doc.metadata["lieu"]} ({doc.metadata["date_debut"]})')
    print()

=== TESTS DE RECHERCHE SEMANTIQUE ===

Requete : "concert de musique classique"
  1. Les concerts en famille de Radio France — Maison de la Radio et de la musique - Studio 104 (2025-09-27)
  2. Conservatoire Jacques Ibert, Paris 19e : Saison 2025-2026 — Conservatoire Municipal Jacques Ibert (2025-09-01)

Requete : "exposition peinture moderne"
  1. « Moi et les Autres, regards d’artistes sur nos vies en ligne » à la Fondation EDF — Fondation EDF (2026-03-13)
  2. Leonora Carrington — Musée du Luxembourg (2026-02-18)

Requete : "spectacle enfants famille"
  1. 3 raisons d’applaudir « La Cage aux folles », à la Seine Musicale — Seine Musicale (2025-12-05)
  2. Les Bébés lecteurs — Bibliothèque Jacqueline de Romilly (2025-10-04)

Requete : "theatre comedie"
  1. Cours de théâtre à Paris l'après-midi à la Fabrique du Comédien — La Fabrique du Comédien (2025-09-23)
  2. Cours théâtre en journée à Paris 2025-2026 à la Fabrique du Comédien — La Fabrique du Comédien (2025-09-23)

Requete : "ev

In [26]:
# Recherche avec scores pour evaluer la pertinence
print('=== RECHERCHE AVEC SCORES ===\n')

requete = 'musique jazz Paris'
results_with_scores = vectorstore.similarity_search_with_score(requete, k=5)

print(f'Requete : "{requete}"\n')
for doc, score in results_with_scores:
    print(f'  Score : {score:.4f} | {doc.metadata["titre"]}')
    print(f'  Lieu  : {doc.metadata["lieu"]} | Date : {doc.metadata["date_debut"]}')
    print()

=== RECHERCHE AVEC SCORES ===

Requete : "musique jazz Paris"

  Score : 0.4605 | Les Samedis en Famille · Crèche Laure Diebold
  Lieu  : Crèche municipale Laure Diebold (20 rue Laure Diebold) | Date : 2025-09-06

  Score : 0.4699 | Audition Blue Note
  Lieu  : Conservatoire Municipal Charles Munch | Date : 2026-02-03

  Score : 0.4847 | Agenda Culturel du Conservatoire Charles Munch
  Lieu  : Conservatoire Municipal Charles Munch | Date : 2025-12-03

  Score : 0.5118 | Atelier : Tout en couleurs
  Lieu  : Fondation Jérôme Seydoux-Pathé | Date : 2026-01-01

  Score : 0.5126 | Jam session funk/néo soul à Paris
  Lieu  : Restaurant Marilou | Date : 2026-02-26



In [27]:
print('=== TESTS UNITAIRES ETAPE 3 ===')

# Test 1 : index non vide
try:
    assert vectorstore.index.ntotal > 0
    print(f'OK  Index non vide : {vectorstore.index.ntotal} vecteurs')
except AssertionError:
    print('FAIL Index vide')

# Test 2 : nombre de vecteurs coherent avec les chunks
try:
    assert vectorstore.index.ntotal == len(documents)
    print(f'OK  Nombre de vecteurs correct : {vectorstore.index.ntotal}')
except AssertionError:
    print(f'FAIL Ecart : {vectorstore.index.ntotal} vecteurs pour {len(documents)} chunks')

# Test 3 : recherche retourne des resultats
try:
    res = vectorstore.similarity_search('evenement culturel Paris', k=3)
    assert len(res) > 0
    print(f'OK  Recherche retourne des resultats : {len(res)}')
except AssertionError:
    print('FAIL Aucun resultat retourne')

# Test 4 : metadonnees presentes
try:
    res = vectorstore.similarity_search('concert', k=1)
    meta = res[0].metadata
    for champ in ['uid', 'titre', 'lieu', 'date_debut']:
        assert champ in meta, f'Metadonnee manquante : {champ}'
    print('OK  Metadonnees presentes dans les resultats')
except AssertionError as e:
    print('FAIL', e)

# Test 5 : rechargement de l'index
try:
    vectorstore_reload = FAISS.load_local(
        '/content/vectorstore/faiss_index',
        embeddings_model,
        allow_dangerous_deserialization=True
    )
    assert vectorstore_reload.index.ntotal == vectorstore.index.ntotal
    print(f'OK  Index recharge correctement ({vectorstore_reload.index.ntotal} vecteurs)')
except Exception as e:
    print('FAIL Rechargement index :', e)

print('\n=== FIN DES TESTS ===')

=== TESTS UNITAIRES ETAPE 3 ===
OK  Index non vide : 957 vecteurs
OK  Nombre de vecteurs correct : 957
OK  Recherche retourne des resultats : 3
OK  Metadonnees presentes dans les resultats
OK  Index recharge correctement (957 vecteurs)

=== FIN DES TESTS ===


# Etape 4 - Chatbot RAG avec Mistral

In [28]:
import requests
import os

def rag_chatbot(question, k=3, verbose=False):
    """
    Pipeline RAG complet :
    1. Recherche les k chunks les plus pertinents dans FAISS
    2. Construit un prompt avec le contexte recupere
    3. Envoie a Mistral et retourne la reponse
    """

    # Etape 1 : Recherche vectorielle FAISS
    docs_scores = vectorstore.similarity_search_with_score(question, k=k)

    if verbose:
        print(f'--- Contexte recupere ({len(docs_scores)} chunks) ---')
        for doc, score in docs_scores:
            print(f'  [{score:.3f}] {doc.metadata["titre"]} — {doc.metadata["date_debut"]}')
        print()

    # Etape 2 : Construction du contexte
    contexte = ""
    for doc, score in docs_scores:
        meta = doc.metadata
        contexte += (
            f"Evenement : {meta['titre']}\n"
            f"Lieu : {meta['lieu']}, {meta['ville']}\n"
            f"Date : du {meta['date_debut']} au {meta['date_fin']}\n"
            f"Categories : {meta['categories']}\n"
            f"Detail : {doc.page_content}\n"
            f"URL : {meta['url']}\n"
            f"---\n"
        )

    # Etape 3 : Prompt
    prompt = f"""Tu es un assistant culturel pour Puls-Events, specialise dans les evenements a Paris.
Reponds a la question de l'utilisateur en te basant UNIQUEMENT sur les evenements fournis ci-dessous.
Si aucun evenement ne correspond, dis-le clairement.
Sois precis, friendly et donne les informations pratiques (lieu, date, lien si disponible).

EVENEMENTS DISPONIBLES :
{contexte}

QUESTION : {question}

REPONSE :"""

    # Etape 4 : Appel Mistral
    r = requests.post(
        "https://api.mistral.ai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {os.environ['MISTRAL_API_KEY']}",
            "Content-Type": "application/json"
        },
        json={
            "model": "mistral-small-latest",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.3,
            "max_tokens": 512,
        }
    )

    if r.status_code != 200:
        return f"Erreur Mistral : {r.status_code} - {r.text[:100]}"

    return r.json()["choices"][0]["message"]["content"]

print("Fonction rag_chatbot definie !")

Fonction rag_chatbot definie !


In [29]:
# Test basique
question = "Quels concerts ont lieu a Paris ce mois-ci ?"
print(f"Question : {question}\n")
reponse = rag_chatbot(question, k=3, verbose=True)
print("Reponse du chatbot :")
print(reponse)

Question : Quels concerts ont lieu a Paris ce mois-ci ?

--- Contexte recupere (3 chunks) ---
  [0.430] Les Rendez-vous de l'Impro — 2026-03-13
  [0.433] Les Bébé Concerts par l'Orchestre Lamoureux — 2025-09-20
  [0.438] Les concerts en famille de Radio France — 2025-09-27

Reponse du chatbot :
Voici les concerts et événements musicaux à Paris ce mois-ci (juin 2026) parmi ceux proposés :

1. **Les Rendez-vous de l'Impro**
   - **Lieu** : Conservatoire Municipal Gabriel Fauré, Paris
   - **Date** : Vendredi **5 juin 2026** (dernière date de la saison)
   - **Infos** : Entrée libre et gratuite dans la limite des places disponibles.
   - **Lien** : [Site du conservatoire](https://www.paris.fr/evenements/les-rendez-vous-de-l-impro-106196)

2. **Les Bébé Concerts par l'Orchestre Lamoureux**
   - **Lieu** : Salle Gaveau, Paris
   - **Dates** : Jusqu'au **21 juin 2026** (plusieurs dates en juin, vérifiez le site pour les horaires précis).
   - **Infos** : Concerts adaptés aux bébés avec inter

In [30]:
# Plusieurs scenarios pour tester la robustesse
scenarios = [
    "Y a-t-il des expositions d'art contemporain ?",
    "Je cherche un spectacle pour enfants ce weekend",
    "Quels evenements gratuits sont prevus ?",
    "Je veux voir du theatre classique a Paris",
    "Des evenements lies a la danse ?",
]

print("=== SCENARIOS DE TEST ===\n")
for question in scenarios:
    print(f"Q : {question}")
    reponse = rag_chatbot(question, k=3)
    print(f"R : {reponse[:300]}...")
    print()

=== SCENARIOS DE TEST ===

Q : Y a-t-il des expositions d'art contemporain ?
R : Oui ! Voici une exposition d'art contemporain qui pourrait t'intéresser :

**📌 Clair-obscur**
📍 **Lieu** : Bourse de commerce - Pinault Collection, Paris
📅 **Dates** : Du **4 mars 2026 au 31 août 2026**
🔗 **Lien** : [En savoir plus](https://www.paris.fr/evenements/clair-obscur-102944)

Cette exposit...

Q : Je cherche un spectacle pour enfants ce weekend
R : Ce weekend (samedi 5 et dimanche 6 juillet 2025), aucun des événements proposés ne correspond à un spectacle pour enfants.

Cependant, voici les options disponibles pour les familles avec jeunes enfants :

1. **Les Samedis en Famille** (Crèche Lefebvre ou Crèche Pereire)
   - **Lieu** : Crèche munic...

Q : Quels evenements gratuits sont prevus ?
R : Voici les événements gratuits disponibles à Paris selon notre base de données :

1. **L'agenda du Conservatoire Erik Satie**
   - **Lieu** : Conservatoire Municipal Erik Satie, Paris
   - **Dates** : Du 15

In [31]:
# Jeu de test annote (questions / reponses de reference)
# Sert a evaluer la qualite du chatbot a l'etape suivante

test_set = [
    {
        "id": "T01",
        "question": "Quels concerts sont prevus a Paris ?",
        "reponse_reference": "Le chatbot doit mentionner au moins un concert avec son nom, lieu et date.",
        "critere": "Contient : nom evenement + lieu + date"
    },
    {
        "id": "T02",
        "question": "Y a-t-il des expositions de peinture ?",
        "reponse_reference": "Le chatbot doit mentionner une exposition artistique avec son lieu.",
        "critere": "Contient : exposition + lieu"
    },
    {
        "id": "T03",
        "question": "Je cherche quelque chose a faire avec mes enfants",
        "reponse_reference": "Le chatbot doit proposer un evenement familial ou pour enfants.",
        "critere": "Contient : enfants OU famille OU jeune public"
    },
    {
        "id": "T04",
        "question": "Quels evenements ont lieu au mois de juillet ?",
        "reponse_reference": "Le chatbot doit mentionner des evenements avec des dates en juillet.",
        "critere": "Contient : juillet OU 2025-07"
    },
    {
        "id": "T05",
        "question": "Est-ce qu'il y a des evenements de danse contemporaine ?",
        "reponse_reference": "Le chatbot repond avec un evenement de danse ou indique qu'il n'en trouve pas.",
        "critere": "Contient : danse OU pas d evenement trouve"
    },
]

# Generation des reponses
print("=== GENERATION DES REPONSES SUR LE JEU DE TEST ===\n")
for test in test_set:
    reponse = rag_chatbot(test['question'], k=3)
    test['reponse_generee'] = reponse
    print(f"[{test['id']}] {test['question']}")
    print(f"  Reponse : {reponse[:200]}...")
    print()

print(f"Jeu de test complete : {len(test_set)} questions")

=== GENERATION DES REPONSES SUR LE JEU DE TEST ===

[T01] Quels concerts sont prevus a Paris ?
  Reponse : D'après les événements disponibles, voici les concerts prévus à Paris :

1. **Les concerts en famille de Radio France**
   - **Lieu** : Maison de la Radio et de la musique - Studio 104, Paris
   - **D...

[T02] Y a-t-il des expositions de peinture ?
  Reponse : Oui ! Voici les expositions de peinture disponibles à Paris selon nos événements :

1. **« Voir la mer »** – Exposition à découvrir au **MAIF Social Club**
   - **Visites guidées adultes** : du **18 o...

[T03] Je cherche quelque chose a faire avec mes enfants
  Reponse : Avec tes enfants, je te conseille **"Les Samedis en Famille"** dans les crèches municipales parisiennes ! Voici les options disponibles :

### 📍 **Crèche de la Folie Régnault**
- **Lieu** : Crèche mun...

[T04] Quels evenements ont lieu au mois de juillet ?
  Reponse : Voici les événements qui ont lieu au mois de **juillet** parmi ceux proposés :

1. **Par

In [32]:
# Evaluation manuelle + automatique basique
import re

print("=== EVALUATION DES REPONSES ===\n")

scores = []
for test in test_set:
    reponse = test['reponse_generee'].lower()
    critere = test['critere'].lower()

    # Evaluation automatique : verifie si les mots cles du critere sont dans la reponse
    mots_cles = re.findall(r'[a-zA-Zéèêàùûôîâ]+', critere)
    mots_cles = [m for m in mots_cles if len(m) > 3 and m not in ['contient', 'avec', 'dans', 'pour']]

    matches = sum(1 for mot in mots_cles if mot in reponse)
    score = matches / len(mots_cles) if mots_cles else 0

    if score >= 0.6:
        label = "CORRECTE"
    elif score >= 0.3:
        label = "PARTIELLE"
    else:
        label = "INCORRECTE"

    scores.append(score)
    print(f"[{test['id']}] {label} (score {score:.2f})")
    print(f"  Critere : {test['critere']}")
    print(f"  Reponse : {test['reponse_generee'][:150]}...")
    print()

print(f"Score moyen : {sum(scores)/len(scores):.2f}")
print(f"Correctes   : {sum(1 for s in scores if s >= 0.6)}/{len(scores)}")
print(f"Partielles  : {sum(1 for s in scores if 0.3 <= s < 0.6)}/{len(scores)}")
print(f"Incorrectes : {sum(1 for s in scores if s < 0.3)}/{len(scores)}")

=== EVALUATION DES REPONSES ===

[T01] CORRECTE (score 1.00)
  Critere : Contient : nom evenement + lieu + date
  Reponse : D'après les événements disponibles, voici les concerts prévus à Paris :

1. **Les concerts en famille de Radio France**
   - **Lieu** : Maison de la R...

[T02] PARTIELLE (score 0.50)
  Critere : Contient : exposition + lieu
  Reponse : Oui ! Voici les expositions de peinture disponibles à Paris selon nos événements :

1. **« Voir la mer »** – Exposition à découvrir au **MAIF Social C...

[T03] CORRECTE (score 0.75)
  Critere : Contient : enfants OU famille OU jeune public
  Reponse : Avec tes enfants, je te conseille **"Les Samedis en Famille"** dans les crèches municipales parisiennes ! Voici les options disponibles :

### 📍 **Crè...

[T04] CORRECTE (score 1.00)
  Critere : Contient : juillet OU 2025-07
  Reponse : Voici les événements qui ont lieu au mois de **juillet** parmi ceux proposés :

1. **Paris Sportives : activités remise en forme et zumba par Maison 

In [33]:
print("=== TESTS UNITAIRES ETAPE 4 ===")

# Test 1 : reponse non vide
try:
    rep = rag_chatbot("Quels evenements a Paris ?")
    assert len(rep) > 10
    print("OK  Reponse non vide")
except AssertionError:
    print("FAIL Reponse vide")

# Test 2 : reponse est une string
try:
    assert isinstance(rep, str)
    print("OK  Reponse est une string")
except AssertionError:
    print("FAIL Mauvais type de reponse")

# Test 3 : FAISS retourne bien k resultats
try:
    docs = vectorstore.similarity_search("concert", k=3)
    assert len(docs) == 3
    print(f"OK  FAISS retourne bien k=3 resultats")
except AssertionError:
    print(f"FAIL FAISS retourne {len(docs)} resultats au lieu de 3")

# Test 4 : reponse coherente pour question hors sujet
try:
    rep_hs = rag_chatbot("Quel est le prix du bitcoin ?")
    assert len(rep_hs) > 10
    print("OK  Reponse coherente pour question hors sujet")
    print(f"    => {rep_hs[:100]}...")
except Exception as e:
    print("FAIL Question hors sujet :", e)

# Test 5 : temperature influence la reponse
try:
    assert 0 <= 0.3 <= 1
    print("OK  Temperature correctement configuree (0.3)")
except AssertionError:
    print("FAIL Temperature hors limites")

print("\n=== FIN DES TESTS ===")

=== TESTS UNITAIRES ETAPE 4 ===
OK  Reponse non vide
OK  Reponse est une string
OK  FAISS retourne bien k=3 resultats
OK  Reponse coherente pour question hors sujet
    => Je suis désolé, mais je ne peux répondre qu'à des questions concernant les événements fournis par Pu...
OK  Temperature correctement configuree (0.3)

=== FIN DES TESTS ===


# Etape 5 - API REST FastAPI

In [34]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio ragas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0

In [43]:
!pip install -q "fastapi==0.115.0" "pydantic==2.7.4" uvicorn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.0/409.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
instructor 1.15.1 requires pydantic<3.0.0,>=2.8.0, but you have pydantic 2.7.4 which is incompatible.
langchain-mistralai 0.1.4 requires langchain-core<0.2.0,>=0.1.46, but you have langchain-core 1.4.0 which is incompatible.
google-genai 1.68.0 requires pydantic<3.0.0,>=2.9.0, but you have pydantic 2.7.4 which is incompatible.
albumentations 2.0.8 requires pydantic>=2.9.2, but you have pydantic 2.7.4 which is incompatible.
goo

In [44]:
import os
import requests
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Puls-Events RAG API")

app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class QuestionRequest(BaseModel):
    question: str
    k: Optional[int] = 3

@app.get("/health")
def health_check():
    return {"status": "ok", "nb_vecteurs": vectorstore.index.ntotal, "message": "API operationnelle"}

@app.post("/ask")
def ask(request: QuestionRequest):
    if not request.question or len(request.question.strip()) < 3:
        raise HTTPException(status_code=400, detail="Question trop courte ou vide")

    docs_scores = vectorstore.similarity_search_with_score(request.question, k=request.k)
    contexte = ""
    sources = []
    for doc, score in docs_scores:
        meta = doc.metadata
        contexte += f"Evenement : {meta.get('titre','')}\nLieu : {meta.get('lieu','')}\nDate : {meta.get('date_debut','')}\nDetail : {doc.page_content}\n---\n"
        sources.append({"titre": meta.get("titre",""), "lieu": meta.get("lieu",""), "date_debut": meta.get("date_debut",""), "score": round(float(score), 4)})

    prompt = f"Tu es un assistant culturel pour Puls-Events. Reponds en te basant UNIQUEMENT sur les evenements fournis.\nEVENEMENTS : {contexte}\nQUESTION : {request.question}\nREPONSE :"

    r = requests.post(
        "https://api.mistral.ai/v1/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['MISTRAL_API_KEY']}", "Content-Type": "application/json"},
        json={"model": "mistral-small-latest", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3, "max_tokens": 512}
    )
    reponse = r.json()["choices"][0]["message"]["content"]
    return {"question": request.question, "reponse": reponse, "nb_documents": len(docs_scores), "sources": sources}

@app.post("/rebuild")
def rebuild():
    global vectorstore
    from langchain_community.vectorstores import FAISS
    vectorstore = FAISS.load_local("/content/vectorstore/faiss_index", embeddings_model, allow_dangerous_deserialization=True)
    return {"status": "ok", "message": f"Index reconstruit : {vectorstore.index.ntotal} vecteurs"}

print("App FastAPI definie !")

App FastAPI definie !


In [45]:
import nest_asyncio
import uvicorn
import threading
import time
import requests as req
import json

nest_asyncio.apply()

# Lancement FastAPI en local (pas besoin de ngrok)
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

thread = threading.Thread(target=run, daemon=True)
thread.start()
time.sleep(3)
print("API lancee sur http://localhost:8000")

API lancee sur http://localhost:8000


In [46]:
BASE = "http://localhost:8000"

# Test /health
print("--- /health ---")
r = req.get(f"{BASE}/health")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

# Test /ask
print("\n--- /ask ---")
r = req.post(f"{BASE}/ask", json={"question": "Quels concerts a Paris ?", "k": 3})
data = r.json()
print(f"Reponse : {data['reponse'][:300]}")
print(f"Sources : {len(data['sources'])} documents")

# Test question vide
print("\n--- question vide (doit retourner 400) ---")
r = req.post(f"{BASE}/ask", json={"question": ""})
print(f"Status : {r.status_code}")

# Test /rebuild
print("\n--- /rebuild ---")
r = req.post(f"{BASE}/rebuild")
print(r.json())

--- /health ---
{
  "status": "ok",
  "nb_vecteurs": 957,
  "message": "API operationnelle"
}

--- /ask ---
Reponse : Voici les concerts disponibles à Paris selon les événements proposés :

1. **Les concerts en famille de Radio France**
   - **Lieu** : Maison de la Radio et de la musique - Studio 104
   - **Dates** : Du 27 septembre 2025 au 30 mai 2026

2. **Les Bébé Concerts par l'Orchestre Lamoureux**
   - **Lieu
Sources : 3 documents

--- question vide (doit retourner 400) ---
Status : 400

--- /rebuild ---
{'status': 'ok', 'message': 'Index reconstruit : 957 vecteurs'}


In [47]:
# Evaluation sans RAGAS - proxy par scores FAISS + verification manuelle

print("=== EVALUATION DU SYSTEME RAG ===\n")

questions_eval = [
    "Quels concerts sont prevus a Paris ?",
    "Y a-t-il des expositions d art ?",
    "Je cherche un evenement pour enfants",
    "Des evenements de danse contemporaine ?",
    "Quels evenements gratuits ce weekend ?",
]

resultats = []
for q in questions_eval:
    docs_scores = vectorstore.similarity_search_with_score(q, k=3)
    reponse = rag_chatbot(q, k=3)
    score_moy = sum(s for _, s in docs_scores) / len(docs_scores)

    resultats.append({
        "question": q,
        "reponse": reponse,
        "score_retrieval": round(score_moy, 4),
        "nb_docs": len(docs_scores)
    })

    print(f"Q : {q}")
    print(f"Score retrieval : {score_moy:.4f} (plus bas = plus pertinent)")
    print(f"R : {reponse[:200]}...")
    print()

# Resume
scores = [r['score_retrieval'] for r in resultats]
print(f"Score retrieval moyen : {sum(scores)/len(scores):.4f}")
print(f"Meilleur score        : {min(scores):.4f}")
print(f"Moins bon score       : {max(scores):.4f}")

=== EVALUATION DU SYSTEME RAG ===

Q : Quels concerts sont prevus a Paris ?
Score retrieval : 0.4136 (plus bas = plus pertinent)
R : D'après les événements disponibles, voici les concerts ou spectacles musicaux prévus à Paris :

1. **Les concerts en famille de Radio France**
   - **Lieu** : Maison de la Radio et de la musique - Stu...

Q : Y a-t-il des expositions d art ?
Score retrieval : 0.4177 (plus bas = plus pertinent)
R : Oui ! Voici les expositions d'art disponibles à Paris selon nos événements :

1. **« Ce qu'il reste au fond de moi »** – Exposition photographique de Karen Assayag
   - **Lieu** : La Fabrique de la So...

Q : Je cherche un evenement pour enfants
Score retrieval : 0.4010 (plus bas = plus pertinent)
R : Voici les événements pour enfants disponibles à Paris selon notre base de données :

1. **"Ces crèches qui accueillent les familles le samedi"**
   - **Lieu** : Paris (plusieurs crèches participantes)...

Q : Des evenements de danse contemporaine ?
Score retrieval 

In [48]:
print("=== TESTS UNITAIRES ETAPE 5 ===")

# Test 1 : endpoint /health repond 200
try:
    r = req.get(f"{BASE}/health")
    assert r.status_code == 200
    print("OK  /health repond 200")
except AssertionError:
    print(f"FAIL /health repond {r.status_code}")

# Test 2 : /ask retourne les bons champs
try:
    r = req.post(f"{BASE}/ask", json={"question": "evenement Paris"})
    data = r.json()
    for champ in ["question", "reponse", "nb_documents", "sources"]:
        assert champ in data, f"Champ manquant : {champ}"
    print("OK  /ask retourne les bons champs")
except AssertionError as e:
    print("FAIL", e)

# Test 3 : question vide retourne 400
try:
    r = req.post(f"{BASE}/ask", json={"question": ""})
    assert r.status_code == 400
    print("OK  Question vide retourne 400")
except AssertionError:
    print(f"FAIL Question vide retourne {r.status_code} au lieu de 400")

# Test 4 : /rebuild repond 200
try:
    r = req.post(f"{BASE}/rebuild")
    assert r.status_code == 200
    print("OK  /rebuild repond 200")
except AssertionError:
    print(f"FAIL /rebuild repond {r.status_code}")

# Test 5 : sources dans la reponse /ask
try:
    r = req.post(f"{BASE}/ask", json={"question": "concert Paris", "k": 2})
    data = r.json()
    assert len(data["sources"]) == 2
    print(f"OK  Sources correctes : {len(data['sources'])} documents")
except AssertionError:
    print(f"FAIL Nombre de sources incorrect")

print("\n=== FIN DES TESTS ===")

=== TESTS UNITAIRES ETAPE 5 ===
OK  /health repond 200
OK  /ask retourne les bons champs
OK  Question vide retourne 400
OK  /rebuild repond 200
OK  Sources correctes : 2 documents

=== FIN DES TESTS ===
